In [0]:
import sys

sys.path.append(
    "/Workspace/Users/szylka17@gmail.com/aws-telecom-data-pipeline/package"
)

from transformations.cleaning import clean_measurements
from transformations.features import add_features


BRONZE_TABLE = "workspace.default.telecom_bronze"
SILVER_TABLE = "workspace.default.telecom_silver"
SOURCE_PATH = "s3://YOUR_BUCKET_NAME/bronze/network_measurements.csv"


# Read raw data
df_bronze = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(SOURCE_PATH)
)

# Save Bronze
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(BRONZE_TABLE)
)

# Transform Bronze → Silver
df_clean = clean_measurements(df_bronze)
df_silver = add_features(df_clean)

# Save Silver
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_TABLE)
)

print("Bronze → Silver completed successfully")